# Neural network and pytorch basics

### Some code imports first

In [ ]:
import plotly.graph_objects as go
import numpy as np
import math
import plotly.io as pio
pio.renderers.default = "notebook"

In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

# AI is just curve-fitting!

Consider the following data(observations).

We want to have "model" that given data `x` can prooduce predictions `y`

In [ ]:
import numpy as np
import plotly.graph_objects as go

rng = np.random.default_rng(seed=42)

n_samples = 25
x = np.linspace(-3.0, 3.0, n_samples)

a = 1.5
b = 0.8
sigma = 1.0

y_true = a * x + b * x**2
y = y_true + rng.normal(0.0, sigma, size=n_samples)

fig = go.Figure()

# Trace 0: Noisy samples
fig.add_trace(go.Scatter(x=x, y=y, mode="markers", name="noisy samples",
                         marker=dict(size=6, opacity=0.7)))

# Trace 1: True function -> HIDDEN BY DEFAULT
fig.add_trace(go.Scatter(x=x, y=y_true, mode="lines", name="true function",
                         line=dict(color="crimson", width=2),
                         visible="legendonly")) # <--- Changed here

# Define the custom toggle buttons
updatemenus = [
    dict(
        type="dropdown",
        direction="down",
        active=1,        # <--- Changed to 1 so "Hide True Function" is selected by default
        x=0.1,
        y=1.15,
        buttons=list([
            dict(
                label="Show True Function",
                method="update",
                args=[{"visible": [True, True]}], 
            ),
            dict(
                label="Hide True Function",
                method="update",
                args=[{"visible": [True, "legendonly"]}], # <--- Keeps legend behavior consistent
            )
        ]),
    )
]

fig.update_layout(
    title="Synthetic data: linear + quadratic + noise",
    xaxis_title="x",
    yaxis_title="y",
    template="plotly_white",
    width=700, height=450,
    updatemenus=updatemenus
)

fig.show()

## Traditional curve fitting

Before reaching for a neural network, let's try the simplest possible model: a **straight line**.

$$f(x) = a \cdot x + b$$

Looking at the data you can probably tell that a line won't capture the curvature — but starting simple has two big payoffs:

1. With only **two parameters**, we can actually *visualize* the loss as a surface in 3D and literally see what optimization is doing.
2. The math stays clean enough to derive by hand, so the link between the formulas and the picture is obvious.

We'll come back to a more flexible model later.

In [ ]:
def linear(x, a, b):
    """Our model: y = a*x + b. Slope a, intercept b."""
    return a * x + b

### Fit by hand

Before letting the computer find the parameters, let's get a feel for what each one does. Drag the sliders to change the slope $a$ and intercept $b$ — the orange line updates live. Try to make it pass through the cloud of points as well as you can.

The **MSE** (mean-squared error) printed above the plot tells you how well your guess matches the data: lower is better.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

x_dense = np.linspace(x.min(), x.max(), 200)

def residual_segments(x, y, y_pred):
    """Build x/y arrays that draw a vertical line from each (x_i, y_i) to (x_i, y_pred_i).
    None separators make each pair render as a disconnected segment in a single trace."""
    n = len(x)
    xs = np.empty(3 * n, dtype=object)
    ys = np.empty(3 * n, dtype=object)
    xs[0::3] = x;       ys[0::3] = y
    xs[1::3] = x;       ys[1::3] = y_pred
    xs[2::3] = None;    ys[2::3] = None
    return xs, ys

fit_fig = go.FigureWidget()

# Trace 0: residual lines (drawn first so they sit behind points/model)
fit_fig.add_trace(go.Scatter(x=[], y=[], mode="lines", name="residuals",
                             line=dict(color="rgba(120,120,120,0.5)", width=1),
                             hoverinfo="skip"))
# Trace 1: data
fit_fig.add_trace(go.Scatter(x=x, y=y, mode="markers", name="data",
                             marker=dict(size=6, opacity=0.8)))
# Trace 2: model line
fit_fig.add_trace(go.Scatter(x=x_dense, y=linear(x_dense, 0.0, 0.0),
                             mode="lines", name="model",
                             line=dict(color="orange", width=3)))

fit_fig.update_layout(
    title="MSE: —",
    xaxis_title="x", yaxis_title="y",
    template="plotly_white",
    width=750, height=480,
)

def update_fit(a_, b_):
    y_pred_dense = linear(x_dense, a_, b_)
    y_pred = linear(x, a_, b_)
    mse = float(np.mean((y - y_pred) ** 2))
    rx, ry = residual_segments(x, y, y_pred)
    with fit_fig.batch_update():
        fit_fig.data[0].x = rx
        fit_fig.data[0].y = ry
        fit_fig.data[2].y = y_pred_dense
        fit_fig.layout.title.text = f"MSE: {mse:.3f}"

slider_kwargs = dict(continuous_update=True, readout_format=".2f")
a_slider = widgets.FloatSlider(value=0.0, min=-1.0, max=4.0, step=0.05, description="a (slope)",     **slider_kwargs)
b_slider = widgets.FloatSlider(value=0.0, min=-3.0, max=6.0, step=0.05, description="b (intercept)", **slider_kwargs)

ui = widgets.VBox([a_slider, b_slider])
out = widgets.interactive_output(update_fit, {"a_": a_slider, "b_": b_slider})

display(ui, fit_fig, out)

## From "fit by hand" to "let the math do it"

Tuning sliders works, but it's slow and we never really know if we've found the *best* fit. Let's formalize what "best" means and let calculus find it for us.

### 1. The loss function

For each observation $(x_i, y_i)$ our model predicts $\hat{y}_i = f(x_i; a, b) = a x_i + b$. The **residual** is the gap we just drew as vertical lines:

$$r_i = y_i - \hat{y}_i$$

To summarize all residuals in a single number we square them (so positives and negatives don't cancel, and larger errors are penalized more) and average:

$$\mathcal{L}(a, b) = \frac{1}{N} \sum_{i=1}^{N} \big(y_i - (a x_i + b)\big)^2$$

This is the **mean squared error** — exactly the value printed in the plot title above. It's a function of the two parameters; the data is fixed. Our goal: find $(a, b)$ that make $\mathcal{L}$ as small as possible.

### 2. Derivatives — which way is "downhill"?

If we wiggle one parameter, say $a$, how does the loss change? That's exactly what a partial derivative tells us:

$$\frac{\partial \mathcal{L}}{\partial a} = \frac{1}{N} \sum_{i=1}^{N} -2 x_i \big(y_i - (a x_i + b)\big)$$

The sign tells us the direction: if $\partial \mathcal{L} / \partial a > 0$, increasing $a$ makes the loss worse, so we should decrease it. The magnitude tells us how *steeply* the loss changes — a big derivative means a small step in $a$ moves the loss a lot.

The same logic for the intercept:

$$\frac{\partial \mathcal{L}}{\partial b} = \frac{1}{N} \sum_{i=1}^{N} -2 \big(y_i - (a x_i + b)\big)$$

### 3. The gradient — packaging the derivatives

Stacking both partials into a vector gives us the **gradient**:

$$\nabla \mathcal{L} = \begin{bmatrix} \partial \mathcal{L} / \partial a \\ \partial \mathcal{L} / \partial b \end{bmatrix}$$

The gradient is a vector in parameter space that points in the direction of **steepest *increase*** of the loss. So the direction of steepest *decrease* is just $-\nabla \mathcal{L}$.

### 4. Gradient descent

That gives us a recipe: repeatedly take a small step in the direction that reduces the loss the most.

$$\begin{bmatrix} a \\ b \end{bmatrix} \;\leftarrow\; \begin{bmatrix} a \\ b \end{bmatrix} - \eta \cdot \nabla \mathcal{L}$$

The scalar $\eta$ (the **learning rate**) controls how big each step is. Too small → we crawl. Too large → we overshoot the minimum and may even diverge.

This single idea — *write down a loss, compute its gradient, take a step downhill* — is the engine that trains every neural network in this repo, including GPT.

Since we now only have **two** parameters, we can actually *see* this geometrically: the loss $\mathcal{L}(a, b)$ is a surface above the $(a, b)$ plane, and gradient descent is a ball rolling downhill on that surface. Let's plot it.

### Visualizing the loss surface

We sweep $a$ and $b$ over a grid, evaluate $\mathcal{L}(a, b)$ at each point, and plot the result. The red dot marks the **global minimum** — the unique $(a^*, b^*)$ where the loss is smallest. (For linear least-squares this has a closed-form solution, so we don't actually need gradient descent here. But the picture generalizes: every neural network has a loss landscape like this, just in millions of dimensions instead of two.)

In [ ]:
# Build a grid over the (a, b) parameter space
a_grid = np.linspace(-1.0, 4.0, 80)
b_grid = np.linspace(-3.0, 6.0, 80)
A_mesh, B_mesh = np.meshgrid(a_grid, b_grid)

# Vectorized loss: for every grid point, compute MSE against the data
# y_pred has shape (n_b, n_a, n_samples)
y_pred_grid = A_mesh[..., None] * x + B_mesh[..., None]
loss_grid = np.mean((y - y_pred_grid) ** 2, axis=-1)

# Analytical optimum via ordinary least squares: y = a*x + b
X_design = np.column_stack([x, np.ones_like(x)])
(a_opt, b_opt), *_ = np.linalg.lstsq(X_design, y, rcond=None)
loss_opt = float(np.mean((y - (a_opt * x + b_opt)) ** 2))
print(f"Optimal linear fit:  a = {a_opt:.3f},  b = {b_opt:.3f},  MSE = {loss_opt:.3f}")

surface_fig = go.Figure(data=[
    go.Surface(x=a_grid, y=b_grid, z=loss_grid,
               colorscale="Viridis", opacity=0.9,
               colorbar=dict(title="MSE"), name="loss"),
    go.Scatter3d(x=[a_opt], y=[b_opt], z=[loss_opt],
                 mode="markers+text",
                 marker=dict(size=6, color="red"),
                 text=["minimum"], textposition="top center",
                 name="optimum"),
])
surface_fig.update_layout(
    title="Loss surface  L(a, b)  =  (1/N) Σ (y - (a·x + b))²",
    scene=dict(
        xaxis_title="a (slope)",
        yaxis_title="b (intercept)",
        zaxis_title="L(a, b)",
        camera=dict(eye=dict(x=1.6, y=1.6, z=1.0)),
    ),
    width=800, height=600,
)
surface_fig.show()

## What if the model is non-linear?

A clean convex bowl is the *exception*, not the rule. As soon as the model is non-linear in its parameters — which is the case for **every neural network** — the loss surface starts growing hills, valleys, ridges, saddles, and **local minima**.

Let's see this with the simplest non-linear two-parameter model:

$$f(x; a, b) = a \cdot \sin(b \cdot x)$$

Here $a$ is the amplitude and $b$ is the frequency. Because $\sin$ is periodic, lots of different $(a, b)$ pairs will fit the data *partially* well — each one a potential trap for gradient descent.

To keep the experiment honest, we'll generate fresh data from this same family (so the true parameters *are* recoverable in principle) and then look at what the loss surface actually looks like.

In [ ]:
def sine_model(x, a, b):
    """Non-linear model: y = a * sin(b * x). a = amplitude, b = frequency."""
    return a * np.sin(b * x)

# Fresh synthetic data drawn from the sine family|
rng_sin = np.random.default_rng(seed=7)
a_true_sin = 2.0
b_true_sin = 1.7
sigma_sin = 0.3

x_sin = np.linspace(-4.0, 4.0, 60)
y_sin_true = sine_model(x_sin, a_true_sin, b_true_sin)
y_sin = y_sin_true + rng_sin.normal(0.0, sigma_sin, size=x_sin.size)

x_sin_dense = np.linspace(x_sin.min(), x_sin.max(), 400)

fig_sin_data = go.Figure()
fig_sin_data.add_trace(go.Scatter(x=x_sin, y=y_sin, mode="markers",
                                  name="data", marker=dict(size=6, opacity=0.8)))
fig_sin_data.add_trace(go.Scatter(x=x_sin_dense,
                                  y=sine_model(x_sin_dense, a_true_sin, b_true_sin),
                                  mode="lines",
                                  name=f"true: a={a_true_sin}, b={b_true_sin}",
                                  line=dict(color="crimson", width=2),
                                  visible="legendonly"))
fig_sin_data.update_layout(
    title="Periodic synthetic data",
    xaxis_title="x", yaxis_title="y",
    template="plotly_white", width=700, height=420,
)
fig_sin_data.show()

### Try fitting it by hand

Same exercise as before — drag the sliders for amplitude $a$ and frequency $b$ to fit the points. But now you'll notice you can land on a "pretty good" fit, where the MSE is low but not optimal — and small slider moves only make it *worse*. That's a **local minimum**. Larger slider moves can find better fits elsewhere on the surface, but pure gradient descent (small steps downhill) would get stuck.

In [ ]:
sin_fit_fig = go.FigureWidget()
sin_fit_fig.add_trace(go.Scatter(x=[], y=[], mode="lines", name="residuals",
                                 line=dict(color="rgba(120,120,120,0.5)", width=1),
                                 hoverinfo="skip"))
sin_fit_fig.add_trace(go.Scatter(x=x_sin, y=y_sin, mode="markers", name="data",
                                 marker=dict(size=6, opacity=0.8)))
sin_fit_fig.add_trace(go.Scatter(x=x_sin_dense,
                                 y=sine_model(x_sin_dense, 1.0, 1.0),
                                 mode="lines", name="model",
                                 line=dict(color="orange", width=3)))
sin_fit_fig.update_layout(
    title="MSE: —",
    xaxis_title="x", yaxis_title="y",
    template="plotly_white",
    width=750, height=480,
)

def update_sin_fit(a_, b_):
    y_pred_dense = sine_model(x_sin_dense, a_, b_)
    y_pred = sine_model(x_sin, a_, b_)
    mse = float(np.mean((y_sin - y_pred) ** 2))
    rx, ry = residual_segments(x_sin, y_sin, y_pred)
    with sin_fit_fig.batch_update():
        sin_fit_fig.data[0].x = rx
        sin_fit_fig.data[0].y = ry
        sin_fit_fig.data[2].y = y_pred_dense
        sin_fit_fig.layout.title.text = f"MSE: {mse:.3f}"

slider_kwargs = dict(continuous_update=True, readout_format=".2f")
a_slider_sin = widgets.FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.02,
                                   description="a (amp)",  **slider_kwargs)
b_slider_sin = widgets.FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.02,
                                   description="b (freq)", **slider_kwargs)

ui_sin  = widgets.VBox([a_slider_sin, b_slider_sin])
out_sin = widgets.interactive_output(update_sin_fit,
                                     {"a_": a_slider_sin, "b_": b_slider_sin})
display(ui_sin, sin_fit_fig, out_sin)

### The loss landscape is no longer a bowl

Sweeping $(a, b)$ over a grid and plotting $\mathcal{L}(a, b)$, we get a **non-convex** surface. Note:

- Two global minima at $(a^*, b^*) \approx (2, 1.7)$ and $(-2, -1.7)$, because $a \sin(b x) = (-a) \sin(-b x)$ — the model is invariant under a joint sign flip.
- Many *local* minima where the model's frequency is wrong but happens to line up with some of the data peaks.
- Saddle ridges where the model becomes flat ($b \approx 0$ or $a \approx 0$).

This is the picture you should keep in your head when you read about "training a neural network": there's a high-dimensional landscape, gradient descent is a local procedure, and where you start matters.

In [ ]:
# Grid sweep over the (a, b) parameter space for the sine model
a_grid_sin = np.linspace(-3.0, 3.0, 120)
b_grid_sin = np.linspace(-3.0, 3.0, 120)
A_mesh_sin, B_mesh_sin = np.meshgrid(a_grid_sin, b_grid_sin)

# Vectorized loss: shape (n_b, n_a, n_samples) -> (n_b, n_a)
y_pred_grid_sin = A_mesh_sin[..., None] * np.sin(B_mesh_sin[..., None] * x_sin)
loss_grid_sin = np.mean((y_sin - y_pred_grid_sin) ** 2, axis=-1)

# Locate the two symmetric global minima on the grid
idx_min = np.unravel_index(np.argmin(loss_grid_sin), loss_grid_sin.shape)
a_star = float(A_mesh_sin[idx_min])
b_star = float(B_mesh_sin[idx_min])
loss_star = float(loss_grid_sin[idx_min])
print(f"Grid minimum:  a = {a_star:.3f},  b = {b_star:.3f},  MSE = {loss_star:.3f}")
print(f"True params:   a = {a_true_sin:.3f},  b = {b_true_sin:.3f}")

mesh_line = dict(show=True, color="rgba(255,255,255,0.35)", width=1,
                 start=-3.0, end=3.0, size=0.4)

surface_fig_sin = go.Figure(data=[
    go.Surface(x=a_grid_sin, y=b_grid_sin, z=loss_grid_sin,
               colorscale="Viridis", opacity=0.92,
               colorbar=dict(title="MSE"),
               contours=dict(x=mesh_line, y=mesh_line)),
    go.Scatter3d(x=[a_star, -a_star], y=[b_star, -b_star],
                 z=[loss_star, loss_star],
                 mode="markers+text",
                 marker=dict(size=6, color="red"),
                 text=["global min", "global min (mirror)"],
                 textposition="top center",
                 name="global minima"),
])
surface_fig_sin.update_layout(
    title="Non-convex loss surface for y = a · sin(b · x)",
    scene=dict(
        xaxis_title="a (amplitude)",
        yaxis_title="b (frequency)",
        zaxis_title="L(a, b)",
        camera=dict(eye=dict(x=1.7, y=1.7, z=1.1)),
        aspectmode="auto",
    ),
    autosize=True,
    height=620,
    margin=dict(l=0, r=0, t=40, b=0),
)
surface_fig_sin.show()

### Watching gradient descent in action

Now let's actually *run* the gradient-descent recipe from earlier on this surface. The gradient of $\mathcal{L}(a, b) = \frac{1}{N} \sum_i (y_i - a \sin(b x_i))^2$ works out to:

$$\frac{\partial \mathcal{L}}{\partial a} = \frac{1}{N} \sum_i -2 \sin(b x_i)\big(y_i - a \sin(b x_i)\big)$$

$$\frac{\partial \mathcal{L}}{\partial b} = \frac{1}{N} \sum_i -2 a x_i \cos(b x_i)\big(y_i - a \sin(b x_i)\big)$$

Pick a starting point $(a_0, b_0)$, a learning rate $\eta$, and a number of steps below — the path will be drawn on the surface. Some things to try:

- Start near $(0.5, 2.5)$ and watch GD settle into a **local minimum** (not the global one at $\approx (2, 1.7)$).
- Start near $(1.8, 1.5)$ — close to the truth — and watch it converge cleanly.
- Crank $\eta$ up to $\sim 0.5$ and see the path **diverge** or oscillate.
- Drop $\eta$ to $\sim 0.001$ to see molasses-speed convergence.

In [ ]:
def sine_loss(a, b, x_data, y_data):
    return float(np.mean((y_data - a * np.sin(b * x_data)) ** 2))

def sine_grad(a, b, x_data, y_data):
    r = y_data - a * np.sin(b * x_data)
    dL_da = -2.0 * np.mean(np.sin(b * x_data) * r)
    dL_db = -2.0 * np.mean(a * x_data * np.cos(b * x_data) * r)
    return dL_da, dL_db

def run_gd(a0, b0, eta, n_steps, x_data, y_data):
    a, b = float(a0), float(b0)
    aa = np.empty(n_steps + 1)
    bb = np.empty(n_steps + 1)
    ll = np.empty(n_steps + 1)
    aa[0], bb[0], ll[0] = a, b, sine_loss(a, b, x_data, y_data)
    for i in range(n_steps):
        da, db = sine_grad(a, b, x_data, y_data)
        a -= eta * da
        b -= eta * db
        aa[i + 1], bb[i + 1] = a, b
        ll[i + 1] = sine_loss(a, b, x_data, y_data)
    return aa, bb, ll


gd_fig = go.FigureWidget()

mesh_line = dict(show=True, color="rgba(255,255,255,0.35)", width=1,
                 start=-3.0, end=3.0, size=0.4)

# Trace 0: surface with x/y contour lines acting as a mesh overlay
gd_fig.add_trace(go.Surface(x=a_grid_sin, y=b_grid_sin, z=loss_grid_sin,
                            colorscale="Viridis", opacity=0.65,
                            showscale=True, colorbar=dict(title="MSE"),
                            contours=dict(x=mesh_line, y=mesh_line),
                            name="loss"))
# Trace 1: GD path
gd_fig.add_trace(go.Scatter3d(x=[], y=[], z=[], mode="lines+markers",
                              line=dict(color="orange", width=5),
                              marker=dict(size=3, color="orange"),
                              name="GD path"))
# Trace 2: start marker
gd_fig.add_trace(go.Scatter3d(x=[], y=[], z=[], mode="markers+text",
                              marker=dict(size=7, color="white",
                                          line=dict(color="black", width=1)),
                              text=["start"], textposition="top center",
                              name="start"))
# Trace 3: end marker
gd_fig.add_trace(go.Scatter3d(x=[], y=[], z=[], mode="markers+text",
                              marker=dict(size=7, color="red"),
                              text=["end"], textposition="top center",
                              name="end"))
# Trace 4: global optima (static reference)
gd_fig.add_trace(go.Scatter3d(x=[a_star, -a_star], y=[b_star, -b_star],
                              z=[loss_star, loss_star],
                              mode="markers",
                              marker=dict(size=5, color="red", symbol="diamond"),
                              name="global optima"))

gd_fig.update_layout(
    title="Gradient descent on the sine-model loss surface",
    scene=dict(
        xaxis_title="a (amplitude)",
        yaxis_title="b (frequency)",
        zaxis_title="L(a, b)",
        camera=dict(eye=dict(x=1.7, y=1.7, z=1.1)),
        aspectmode="auto",
    ),
    autosize=True,
    height=620,
    margin=dict(l=0, r=0, t=40, b=0),
)
gd_fig.layout.autosize = True
gd_fig._config = {**(gd_fig._config or {}), "responsive": True}

# Controls — continuous_update=True so dragging triggers re-runs live
slider_layout = widgets.Layout(width="48%")
a0_slider = widgets.FloatSlider(value=0.5, min=-3.0, max=3.0, step=0.05,
                                description="a₀", continuous_update=True,
                                readout_format=".2f", layout=slider_layout)
b0_slider = widgets.FloatSlider(value=2.5, min=-3.0, max=3.0, step=0.05,
                                description="b₀", continuous_update=True,
                                readout_format=".2f", layout=slider_layout)
eta_slider = widgets.FloatLogSlider(value=0.05, base=10, min=-3, max=0, step=0.1,
                                    description="η (lr)", continuous_update=True,
                                    readout_format=".3f", layout=slider_layout)
steps_slider = widgets.IntSlider(value=30, min=1, max=40, step=1,
                                 description="steps", continuous_update=True,
                                 layout=slider_layout)
status = widgets.HTML(value="")

def rerun(*_):
    a0, b0 = a0_slider.value, b0_slider.value
    aa, bb, ll = run_gd(a0, b0, eta_slider.value, steps_slider.value, x_sin, y_sin)
    z_path = ll + 0.05  # lift slightly above the surface
    with gd_fig.batch_update():
        gd_fig.data[1].x, gd_fig.data[1].y, gd_fig.data[1].z = aa, bb, z_path
        gd_fig.data[2].x, gd_fig.data[2].y, gd_fig.data[2].z = [aa[0]],  [bb[0]],  [z_path[0]]
        gd_fig.data[3].x, gd_fig.data[3].y, gd_fig.data[3].z = [aa[-1]], [bb[-1]], [z_path[-1]]
    status.value = (
        f"Start: a₀={aa[0]:.2f}, b₀={bb[0]:.2f}, L₀={ll[0]:.3f} &nbsp;→&nbsp; "
        f"End: a={aa[-1]:.2f}, b={bb[-1]:.2f}, L={ll[-1]:.3f}"
    )

for s in (a0_slider, b0_slider, eta_slider, steps_slider):
    s.observe(rerun, names="value")

controls = widgets.VBox([
    widgets.HBox([a0_slider, b0_slider], layout=widgets.Layout(width="100%")),
    widgets.HBox([eta_slider, steps_slider], layout=widgets.Layout(width="100%")),
    status,
], layout=widgets.Layout(width="100%"))

display(controls, gd_fig)
rerun()  # initial draw